In [1]:
from google.colab import drive
drive.mount('/content/drive')
!unzip -q /content/drive/MyDrive/VPC2/repo.zip -d /content/repo
!unzip -q /content/drive/MyDrive/VPC2/processed.zip -d /content/processed
!pip install -q ultralytics
import torch; print(torch.cuda.is_available())

Mounted at /content/drive
unzip:  cannot find or open /content/drive/MyDrive/VPC2/repo.zip, /content/drive/MyDrive/VPC2/repo.zip.zip or /content/drive/MyDrive/VPC2/repo.zip.ZIP.
unzip:  cannot find or open /content/drive/MyDrive/VPC2/processed.zip, /content/drive/MyDrive/VPC2/processed.zip.zip or /content/drive/MyDrive/VPC2/processed.zip.ZIP.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.7/46.7 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 65.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.5/95.5 kB 10.8 MB/s eta 0:00:00
True


In [2]:
!find /content/processed -name data.yaml
!cat /content/processed/data.yaml

find: ‘/content/processed’: No such file or directory
cat: /content/processed/data.yaml: No such file or directory


Celda A: buscar, descomprimir y arreglar el yaml

In [6]:
import glob, os, sys
from google.colab import drive
drive.mount('/content/drive')

BASE = "/content/drive/MyDrive/Colab Notebooks"

!rm -rf /content/repo /content/processed
!unzip -q -o "{BASE}/repozip.zip" -d /content/repo
!unzip -q -o "{BASE}/processed.zip" -d /content/processed

img_dir = glob.glob('/content/processed/**/images', recursive=True)[0]
DATA_ROOT = os.path.dirname(img_dir)

yaml_text = f"""path: {DATA_ROOT}
train: images/train
val: images/val
test: images/test
names:
  0: BIODEGRADABLE
  1: CARDBOARD
  2: GLASS
  3: METAL
  4: PAPER
  5: PLASTIC
"""
open(f'{DATA_ROOT}/data.yaml', 'w').write(yaml_text)
print("DATA_ROOT:", DATA_ROOT)

REPO_ROOT = os.path.dirname(glob.glob('/content/repo/**/configs', recursive=True)[0])
sys.path.append(os.path.join(REPO_ROOT, 'src'))
os.chdir(REPO_ROOT)
print("REPO_ROOT:", REPO_ROOT)

os.makedirs('/content/drive/MyDrive/VPC2/runs', exist_ok=True)

!ls {DATA_ROOT} {DATA_ROOT}/images

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
DATA_ROOT: /content/processed/processed
REPO_ROOT: /content/repo/repozip
/content/processed/processed:
data.yaml  images  labels

/content/processed/processed/images:
test  train  val


Celda B: entrenar las 2 variantes (tarda entre 1 y 3 horas en total)

In [8]:
from ultralytics import YOLO
from vpc2.data.augmentation import load_policy

policy = load_policy("proposed")
OUT = "/content/drive/MyDrive/VPC2/runs"

model = YOLO("yolov8n.pt")
model.train(data=f"{DATA_ROOT}/data.yaml", epochs=50, imgsz=640,
            batch=16, seed=42, workers=2, project=OUT,
            name="yolov8n_proposed", exist_ok=True,
            **policy.train_kwargs())

Ultralytics 8.4.172 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, augmentations=[{'__version__': '2.0.8', 'transform': {'__class_fullname__': 'RandomBrightnessContrast', 'p': 0.3, 'brightness_by_max': True, 'brightness_limit': (-0.4, 0.05), 'contrast_limit': (0.0, 0.0), 'ensure_safe_range': False}}, {'__version__': '2.0.8', 'transform': {'__class_fullname__': 'MotionBlur', 'p': 0.15, 'allow_shifted': True, 'angle_range': (0.0, 360.0), 'blur_limit': (3, 9), 'direction_range': (-1.0, 1.0)}}, {'__version__': '2.0.8', 'transform': {'__class_fullname__': 'GaussianBlur', 'p': 0.1, 'blur_limit': (0, 0), 'sigma_limit': (0.5, 2.0)}}], auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/process

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3, 4, 5])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x78dad546dcc0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
     

In [9]:
model = YOLO("yolov8s.pt")
model.train(data=f"{DATA_ROOT}/data.yaml", epochs=50, imgsz=640,
            batch=16, seed=42, workers=2, project=OUT,
            name="yolov8s_proposed", exist_ok=True,
            **policy.train_kwargs())

Ultralytics 8.4.172 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, augmentations=[{'__version__': '2.0.8', 'transform': {'__class_fullname__': 'RandomBrightnessContrast', 'p': 0.3, 'brightness_by_max': True, 'brightness_limit': (-0.4, 0.05), 'contrast_limit': (0.0, 0.0), 'ensure_safe_range': False}}, {'__version__': '2.0.8', 'transform': {'__class_fullname__': 'MotionBlur', 'p': 0.15, 'allow_shifted': True, 'angle_range': (0.0, 360.0), 'blur_limit': (3, 9), 'direction_range': (-1.0, 1.0)}}, {'__version__': '2.0.8', 'transform': {'__class_fullname__': 'GaussianBlur', 'p': 0.1, 'blur_limit': (0, 0), 'sigma_limit': (0.5, 2.0)}}], auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/process

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3, 4, 5])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x78db5f13d470>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
     

Celda C: evaluar en test y armar la tabla

In [10]:
import pandas as pd
from ultralytics import YOLO

OUT = "/content/drive/MyDrive/VPC2/runs"
rows, por_clase = [], {}

for name in ["yolov8n_proposed", "yolov8s_proposed"]:
    model = YOLO(f"{OUT}/{name}/weights/best.pt")
    m = model.val(data=f"{DATA_ROOT}/data.yaml", split="test", imgsz=640, plots=False)
    p, r = m.box.mp, m.box.mr
    rows.append({"modelo": name,
                 "mAP50": round(m.box.map50, 4),
                 "mAP50-95": round(m.box.map, 4),
                 "precision": round(p, 4),
                 "recall": round(r, 4),
                 "F1": round(2 * p * r / (p + r), 4),
                 "ms_inferencia": round(m.speed["inference"], 2)})
    por_clase[name] = dict(zip(m.names.values(), m.box.maps.round(4)))

df = pd.DataFrame(rows)
df.to_csv(f"{OUT}/tabla_metricas.csv", index=False)
display(df)
display(pd.DataFrame(por_clase))

Ultralytics 8.4.172 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 72 layers, 3,006,818 parameters, 0 gradients, 8.1 GFLOPs
WARNING ⚠️ val: Slow image access detected (ping: 0.0±0.0 ms, read: 12.5±5.8 MB/s, size: 16.7 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips
val: Scanning /content/processed/processed/labels/test... 1045 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1045/1045 650.5it/s 1.6s
val: New cache created: /content/processed/processed/labels/test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 66/66 6.6it/s 10.1s
                   all       1045       7452      0.715      0.555      0.629      0.443
         BIODEGRADABLE        228       4504      0.767      0.504      0.606      0.345
             CARDBOARD        146        490       0.66       0.44       0.51      0

,modelo,mAP50,mAP50-95,precision,recall,F1,ms_inferencia
0,yolov8n_proposed,0.6293,0.4425,0.7150,0.5547,0.6248,4.09
1,yolov8s_proposed,0.6502,0.4673,0.7429,0.5772,0.6496,9.79


,yolov8n_proposed,yolov8s_proposed
BIODEGRADABLE,0.3450,0.3635
CARDBOARD,0.3867,0.4001
GLASS,0.6093,0.6236
METAL,0.4797,0.5123
PAPER,0.3821,0.4278
PLASTIC,0.4525,0.4767


In [11]:
import time, glob, torch, numpy as np
from ultralytics import YOLO

imgs = sorted(glob.glob(f"{DATA_ROOT}/images/test/*.jpg") + glob.glob(f"{DATA_ROOT}/images/test/*.png"))[:200]
res = {}
for name in ["yolov8n_proposed", "yolov8s_proposed"]:
    model = YOLO(f"{OUT}/{name}/weights/best.pt")
    for p in imgs[:20]:  # calentamiento
        model.predict(p, imgsz=640, verbose=False)
    tiempos = []
    for p in imgs:
        torch.cuda.synchronize(); t = time.perf_counter()
        model.predict(p, imgsz=640, verbose=False)
        torch.cuda.synchronize(); tiempos.append((time.perf_counter() - t) * 1000)
    res[name] = {"ms_medio": round(np.mean(tiempos), 2),
                 "ms_mediana": round(np.median(tiempos), 2),
                 "FPS": round(1000 / np.mean(tiempos), 1)}
res

{'yolov8n_proposed': {'ms_medio': np.float64(10.22),
  'ms_mediana': np.float64(10.02),
  'FPS': np.float64(97.9)},
 'yolov8s_proposed': {'ms_medio': np.float64(18.11),
  'ms_mediana': np.float64(18.07),
  'FPS': np.float64(55.2)}}

In [12]:
import pandas as pd
pd.DataFrame(res).T.to_csv(f"{OUT}/latencia.csv")